# Prédiction de la durée de séjour

## Objectif

L'objectif est de prédire `length_of_stay`, c'est-à-dire la durée du séjour hospitalier en jours.

**Hypothèse métier :** la prédiction est réalisée après le diagnostic initial. Les variables disponibles à ce moment peuvent donc être utilisées. Les variables directement liées à la sortie ou au risque de réadmission sont exclues afin d'éviter une fuite d'information (*data leakage*).

# 1. Imports des librairies et configuration

In [28]:
# Cette cellule centralise les imports et les paramètres afin d'améliorer la lisibilité et la reproductibilité du notebook

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import mlflow
import mlflow.sklearn
import dagshub
import optuna
import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RANDOM_STATE,
    DAGSHUB_OWNER,
    DAGSHUB_REPO,
)

TEST_SIZE = 0.2
CV_FOLDS = 5
RANDOM_STATE = RANDOM_STATE
DAGSHUB_OWNER = DAGSHUB_OWNER
DAGSHUB_REPO = DAGSHUB_REPO
MLFLOW_EXPERIMENT = "length_of_stay_prediction"

print(f"XGBoost version : {xgb.__version__}")
print(f"MLflow version : {mlflow.__version__}")

XGBoost version : 3.2.0
MLflow version : 3.15.2


# 2. Chargement du dataset

In [29]:
# Le jeu de données utilisé est le même dans les deux notebooks : celui-ci provient de Kaggle
DATA_URL = (
    "https://drive.google.com/uc?export=download&"
    "id=1YkRYSkAHeH7nMgO-ums8zbU7paODGs3t"
)

df = pd.read_csv(DATA_URL)

print(f"Nombre de lignes : {len(df)}")
print(f"Nombre de colonnes : {df.shape[1]}")
display(df.head(3))

Nombre de lignes : 8000
Nombre de colonnes : 17


,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
0,P00001,2022-04-14,Spring,66,Male,South,Diabetes,5,6,Interventional,8,6,1,Medicare,Home Health,0.92,1
1,P00002,2021-09-19,Fall,55,Male,South,Diabetes,4,6,Interventional,6,4,3,Private,Home Health,0.88,1
2,P00003,2023-04-12,Spring,69,Female,West,Hypertension,6,8,Medical,9,6,2,Medicare,Skilled Nursing,0.97,1


# 3. Contrôle initial des données

In [30]:
# Cette étape permet de vérifier la structure du jeu de données, les types, les valeurs manquantes et la distribution de la cible.
print(df.info())
print("\nValeurs manquantes :")
print(df.isna().sum())

print("\nDistribution de length_of_stay :")
display(df["length_of_stay"].value_counts().sort_index())

print("\nStatistiques descriptives :")
display(df["length_of_stay"].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 8000 non-null   object 
 1   admission_date             8000 non-null   object 
 2   season                     8000 non-null   object 
 3   age                        8000 non-null   int64  
 4   gender                     8000 non-null   object 
 5   region                     8000 non-null   object 
 6   primary_diagnosis          8000 non-null   object 
 7   comorbidities_count        8000 non-null   int64  
 8   length_of_stay             8000 non-null   int64  
 9   treatment_type             8000 non-null   object 
 10  medications_count          8000 non-null   int64  
 11  followup_visits_last_year  8000 non-null   int64  
 12  prev_readmissions          8000 non-null   int64  
 13  insurance_type             8000 non-null   objec

length_of_stay
3       14
4      166
5      764
6     1342
7     1410
8     1453
9     1287
10     867
11     465
12     163
13      52
14      14
15       3
Name: count, dtype: int64


Statistiques descriptives :


count    8000.000000
mean        7.789125
std         1.930252
min         3.000000
25%         6.000000
50%         8.000000
75%         9.000000
max        15.000000
Name: length_of_stay, dtype: float64

### Distribution de la cible

`length_of_stay` est une variable numérique discrète exprimée en jours. Cette distribution sera également utile pour interpréter les erreurs du modèle : une erreur absolue de 1 jour n'a pas la même portée qu'une erreur de 1 jour sur un séjour moyen de 5 jours ou de 11 jours.

In [31]:
fig = px.histogram(
    df,
    x="length_of_stay",
    nbins=int(df["length_of_stay"].max() - df["length_of_stay"].min() + 1),
    title="Distribution de la durée de séjour",
    labels={
        "length_of_stay": "Durée de séjour (jours)",
        "count": "Nombre de patients",
    },
)
fig.show()

## 4. Définition de la cible et gestion du data leakage

Les variables suivantes sont exclues du modèle :

- `discharge_disposition` : information directement liée à la sortie ;
- `readmission_risk_score` : score construit pour une autre problématique et potentiellement disponible après le séjour ;
- `label` : cible du modèle de réadmission.

`patient_id` et `admission_date` sont également retirés des features car ils ne constituent pas des variables explicatives pertinentes dans le cadre actuel.

**Point important :** cette sélection est définie avant l'entraînement et reste identique pour tous les modèles afin de permettre une comparaison équitable.

In [32]:
TARGET = "length_of_stay"

LEAKAGE_COLUMNS = [
    "discharge_disposition",
    "readmission_risk_score",
    "label",
]

IDENTIFIER_COLUMNS = [
    "patient_id",
    "admission_date",
]

COLUMNS_TO_DROP = LEAKAGE_COLUMNS + IDENTIFIER_COLUMNS

df_ml = df.drop(columns=COLUMNS_TO_DROP)

X = df_ml.drop(columns=TARGET)
y = df_ml[TARGET]

print("Variables explicatives :")
print(X.columns.tolist())
print(f"\nNombre de features : {X.shape[1]}")


Variables explicatives :
['season', 'age', 'gender', 'region', 'primary_diagnosis', 'comorbidities_count', 'treatment_type', 'medications_count', 'followup_visits_last_year', 'prev_readmissions', 'insurance_type']

Nombre de features : 11


## 5. Analyse exploratoire des relations avec la cible

Cette analyse est **exploratoire**. Elle sert à comprendre le jeu de données et à formuler des hypothèses, mais elle ne doit pas servir à sélectionner les variables à partir de l'ensemble des données avant la séparation train/test.

C'est particulièrement important ici car `primary_diagnosis` présente une relation très forte avec `length_of_stay` dans le dataset fourni.


In [33]:
numeric_features_all = X.select_dtypes(include="number").columns

numeric_correlation = (
    df_ml[numeric_features_all]
    .corrwith(df_ml[TARGET])
    .sort_values()
)

display(numeric_correlation.to_frame("correlation"))

,correlation
prev_readmissions,0.146806
followup_visits_last_year,0.163643
age,0.205709
medications_count,0.213912
comorbidities_count,0.243239


### Eta carré pour les variables catégorielles

Pour une variable catégorielle, le coefficient de corrélation de Pearson n'est pas adapté. L'eta carré permet ici de mesurer la proportion de variance de la cible associée aux différences entre groupes.

Cette mesure est utilisée uniquement pour l'analyse exploratoire.

In [34]:
def calculate_eta_squared(data, categorical_column, target_column):
    """Calculate eta squared between a categorical variable and a target."""
    grand_mean = data[target_column].mean()
    total_sum_squares = (
        (data[target_column] - grand_mean) ** 2
    ).sum()

    grouped = data.groupby(categorical_column)[target_column]
    category_means = grouped.mean()
    category_counts = grouped.count()

    between_sum_squares = (
        category_counts * (category_means - grand_mean) ** 2
    ).sum()

    if total_sum_squares == 0:
        return 0.0

    return between_sum_squares / total_sum_squares


categorical_features_all = X.select_dtypes(
    include="object"
).columns

eta_squared = {
    column: calculate_eta_squared(df_ml, column, TARGET)
    for column in categorical_features_all
}

eta_squared = pd.Series(eta_squared).sort_values(ascending=False)
display(eta_squared.to_frame("eta_squared"))

,eta_squared
primary_diagnosis,0.653437
season,0.078522
insurance_type,0.021054
treatment_type,0.000964
region,0.000308
gender,0.000103


Interprétation :

η²(primary_diagnosis) = 0,653
η²(season) = 0,079
η²(insurance_type) = 0,021
les autres variables catégorielles sont quasiment nulles.

Le diagnostic explique donc à lui seul environ 65 % de la variance de length_of_stay dans ton analyse.

### Focus sur `primary_diagnosis`

L'analyse précédente montre que `primary_diagnosis` est la variable catégorielle la plus fortement associée à la durée de séjour. Il est donc pertinent de vérifier les moyennes et la dispersion de la cible selon le diagnostic.

In [35]:
diagnosis_analysis = (
    df_ml.groupby("primary_diagnosis")[TARGET]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
    .round(2)
)

display(diagnosis_analysis)

,mean,std,count
primary_diagnosis,,,
Hypertension,5.82,1.14,1301
Appendicitis,5.88,1.14,534
Diabetes,6.75,1.14,1470
Influenza,6.79,1.13,233
Kidney Disease,7.74,1.12,868
Fracture,7.81,1.14,434
Pneumonia,8.70,1.06,425
COPD,8.83,1.16,952
Stroke,9.74,1.17,883


Les différences entre diagnostics sont très nettes. 
On passe d'environ 5,8 jours à 10,76 jours selon le diagnostic.  

Le diagnostic constitue ainsi le principal déterminant disponible dans les données pour expliquer la durée d'hospitalisation.

In [36]:
fig = px.box(
    df_ml,
    x="primary_diagnosis",
    y=TARGET,
    category_orders={
        "primary_diagnosis": diagnosis_analysis.index.tolist()
    },
    title="Durée de séjour selon le diagnostic principal",
    labels={
        "primary_diagnosis": "Diagnostic principal",
        TARGET: "Durée de séjour (jours)",
    },
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Interprétation :** les moyennes sont nettement différentes selon le diagnostic et les écarts-types restent relativement homogènes. Le diagnostic explique donc une part importante de la variabilité observée dans la cible.

Cette observation justifie la construction d'une baseline plus exigeante qu'une simple moyenne globale : nous comparerons le modèle à une **moyenne de durée par diagnostic calculée uniquement sur le train set**.

## 6. Séparation train/test

La séparation intervient avant toute étape d'apprentissage. Les paramètres du preprocessing et la baseline par diagnostic seront appris uniquement sur `X_train` et `y_train`.

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

numeric_features = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

Train : (6400, 11)
Test  : (1600, 11)


## 7. Preprocessing

Le preprocessing est encapsulé dans un `ColumnTransformer` afin que les opérations d'imputation, de standardisation et d'encodage soient apprises à l'intérieur de chaque entraînement. Cela évite les fuites d'information pendant la validation croisée.

In [38]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

In [39]:
dagshub.init(
    repo_owner=DAGSHUB_OWNER,
    repo_name=DAGSHUB_REPO,
    mlflow=True,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT)

Initialized MLflow to track repo "quentin.seng67/hospital_readmission"

Repository quentin.seng67/hospital_readmission initialized!

<Experiment: artifact_location='mlflow-artifacts:/07ff9153544d4427bba3bf62aacf4946', creation_time=1788263313697, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788263313697, lifecycle_stage='active', name='length_of_stay_prediction', tags={}, trace_location=None, workspace='default'>

## 8. Baselines

Deux niveaux de référence sont utiles :

1. `DummyRegressor` prédit la moyenne globale du train ;
2. la baseline métier prédit la moyenne de `length_of_stay` du diagnostic, calculée sur le train uniquement.

Le second niveau est particulièrement important ici puisque l'analyse exploratoire a montré une forte association entre diagnostic et durée de séjour.

In [40]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

y_dummy_pred = dummy_model.predict(X_test)

dummy_mae = mean_absolute_error(y_test, y_dummy_pred)
dummy_rmse = root_mean_squared_error(
    y_test,
    y_dummy_pred,
)
dummy_r2 = r2_score(y_test, y_dummy_pred)

print(f"DummyRegressor - MAE  : {dummy_mae:.3f} jours")
print(f"DummyRegressor - RMSE : {dummy_rmse:.3f} jours")
print(f"DummyRegressor - R²   : {dummy_r2:.3f}")

DummyRegressor - MAE  : 1.635 jours
DummyRegressor - RMSE : 2.008 jours
DummyRegressor - R²   : -0.003


Le DummyRegressor donne :

- MAE : 1,635 jours
- RMSE : 2,008 jours
- R² : -0,003

Le R² légèrement négatif signifie que prédire systématiquement la moyenne globale est à peine moins bon que la référence utilisée pour calculer le R².

Ce modèle sert surtout de point de comparaison minimal.

Conclusion : Connaître autre chose que la moyenne globale apporte énormément d'information.

In [41]:
diagnosis_means = (
    pd.concat([X_train[["primary_diagnosis"]], y_train], axis=1)
    .groupby("primary_diagnosis")[TARGET]
    .mean()
)

y_diagnosis_baseline = X_test["primary_diagnosis"].map(
    diagnosis_means
)

if y_diagnosis_baseline.isna().any():
    raise ValueError(
        "Un diagnostic présent dans le test est absent du train."
    )

diagnosis_mae = mean_absolute_error(
    y_test,
    y_diagnosis_baseline,
)
diagnosis_rmse = root_mean_squared_error(
    y_test,
    y_diagnosis_baseline,
)
diagnosis_r2 = r2_score(y_test, y_diagnosis_baseline)

print(f"Baseline diagnostic - MAE  : {diagnosis_mae:.3f} jours")
print(f"Baseline diagnostic - RMSE : {diagnosis_rmse:.3f} jours")
print(f"Baseline diagnostic - R²   : {diagnosis_r2:.3f}")

Baseline diagnostic - MAE  : 0.947 jours
Baseline diagnostic - RMSE : 1.204 jours
Baseline diagnostic - R²   : 0.640


La baseline par diagnostic obtient :

- MAE : 0,947 jour
- RMSE : 1,204 jour
- R² : 0,640

Ainsi, le fait de prédire la durée moyenne correspondant au diagnostic permet déjà d'obtenir une erreur moyenne inférieure à un jour.

L'EDA a montré que primary_diagnosis est fortement associé à la durée de séjour. Nous avons donc construit une baseline métier consistant à prédire, pour chaque patient, la durée moyenne observée pour son diagnostic dans le jeu d'entraînement.  
Cette baseline atteint une MAE de 0,947 jour.  
Les modèles de machine learning suivants devront donc démontrer qu'ils apportent une information supplémentaire par rapport au seul diagnostic.

## 9. Validation croisée

Comme dans `readmission_model`, les modèles seront comparés sur le train set avec une validation croisée à 5 folds.

Pour une régression de durée, la métrique principale retenue est la **MAE** (*Mean Absolute Error*) car elle s'interprète directement en jours : une MAE de 0,7 signifie une erreur absolue moyenne d'environ 0,7 jour.

Le RMSE et le R² sont conservés comme métriques complémentaires.

In [42]:
cv = KFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = [
    "neg_mean_absolute_error",
    "neg_root_mean_squared_error",
    "r2",
]

## 10. Régression linéaire

La régression linéaire constitue notre premier modèle supervisé. Elle fournit un point de comparaison simple avant de tester des modèles capables de représenter des interactions et des relations non linéaires.

In [43]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

linear_cv = cross_validate(
    linear_model,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
    return_train_score=True,
)

linear_cv_mae = -linear_cv["test_neg_mean_absolute_error"].mean()
linear_cv_rmse = -linear_cv["test_neg_root_mean_squared_error"].mean()
linear_cv_r2 = linear_cv["test_r2"].mean()

linear_model.fit(X_train, y_train)
y_linear_pred = linear_model.predict(X_test)

linear_test_mae = mean_absolute_error(y_test, y_linear_pred)
linear_test_rmse = root_mean_squared_error(
    y_test,
    y_linear_pred,
)
linear_test_r2 = r2_score(y_test, y_linear_pred)

print("=== Validation croisée ===")
print(f"MAE  : {linear_cv_mae:.3f} jours")
print(f"RMSE : {linear_cv_rmse:.3f} jours")
print(f"R²   : {linear_cv_r2:.3f}")

print("\n=== Test set ===")
print(f"MAE  : {linear_test_mae:.3f} jours")
print(f"RMSE : {linear_test_rmse:.3f} jours")
print(f"R²   : {linear_test_r2:.3f}")

=== Validation croisée ===
MAE  : 0.810 jours
RMSE : 1.014 jours
R²   : 0.718

=== Test set ===
MAE  : 0.847 jours
RMSE : 1.083 jours
R²   : 0.708


### Comparaison à la baseline diagnostic

Cette comparaison est plus informative qu'un simple R² global : elle permet de vérifier si le modèle apporte réellement de l'information supplémentaire par rapport à la connaissance du diagnostic principal.

In [44]:
linear_results = X_test.copy()
linear_results["actual"] = y_test
linear_results["predicted"] = y_linear_pred
linear_results["baseline_pred"] = y_diagnosis_baseline

linear_results["model_error"] = (
    linear_results["actual"] - linear_results["predicted"]
).abs()
linear_results["baseline_error"] = (
    linear_results["actual"] - linear_results["baseline_pred"]
).abs()

linear_by_diagnosis = (
    linear_results.groupby("primary_diagnosis")
    [["model_error", "baseline_error"]]
    .mean()
    .rename(
        columns={
            "model_error": "mae_linear",
            "baseline_error": "mae_baseline",
        }
    )
)

linear_by_diagnosis["gain"] = (
    linear_by_diagnosis["mae_baseline"]
    - linear_by_diagnosis["mae_linear"]
)

display(linear_by_diagnosis.round(3))

,mae_linear,mae_baseline,gain
primary_diagnosis,,,
Appendicitis,0.773,0.861,0.088
COPD,0.896,1.021,0.125
Diabetes,0.854,0.962,0.108
Fracture,0.771,0.847,0.077
Heart Failure,0.681,0.788,0.107
Hypertension,0.857,0.974,0.116
Influenza,0.817,0.858,0.040
Kidney Disease,0.881,0.953,0.072
Pneumonia,0.810,0.899,0.090


In [45]:
fig = px.bar(
    linear_by_diagnosis.reset_index().melt(
        id_vars="primary_diagnosis",
        value_vars=["mae_linear", "mae_baseline"],
        var_name="method",
        value_name="mae",
    ),
    x="primary_diagnosis",
    y="mae",
    color="method",
    barmode="group",
    title="MAE : régression linéaire vs baseline diagnostic",
    labels={
        "primary_diagnosis": "Diagnostic principal",
        "mae": "MAE (jours)",
        "method": "Méthode",
    },
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

La régression linéaire surpasse la baseline basée sur les diagnostics seuls :

0,947 → 0,847 jour de MAE soit environ 0,10 jour de mieux.

Et le R² passe :

0,640 → 0,708

Interprétation : le modèle n'apprend pas uniquement les différences de durée entre diagnostics, les autres variables apportent une information complémentaire.

Conclusion : Le diagnostic explique une grande partie de la durée de séjour, mais la régression linéaire montre que d'autres caractéristiques individuelles permettent d'affiner la prédiction.

## 11. Random Forest

Random Forest permet de tester un modèle non linéaire avant l'optimisation. L'objectif n'est pas encore de rechercher les meilleurs hyperparamètres, mais de mesurer le gain obtenu par rapport à la régression linéaire et à la baseline diagnostic.

In [46]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

rf_cv = cross_validate(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
)

rf_cv_mae = -rf_cv["test_neg_mean_absolute_error"].mean()
rf_cv_rmse = -rf_cv["test_neg_root_mean_squared_error"].mean()
rf_cv_r2 = rf_cv["test_r2"].mean()

random_forest_model.fit(X_train, y_train)
y_rf_pred = random_forest_model.predict(X_test)

rf_test_mae = mean_absolute_error(y_test, y_rf_pred)
rf_test_rmse = root_mean_squared_error(
    y_test,
    y_rf_pred,
)
rf_test_r2 = r2_score(y_test, y_rf_pred)

print("=== Random Forest - validation croisée ===")
print(f"MAE  : {rf_cv_mae:.3f} jours")
print(f"RMSE : {rf_cv_rmse:.3f} jours")
print(f"R²   : {rf_cv_r2:.3f}")

print("\n=== Random Forest - test set ===")
print(f"MAE  : {rf_test_mae:.3f} jours")
print(f"RMSE : {rf_test_rmse:.3f} jours")
print(f"R²   : {rf_test_r2:.3f}")

=== Random Forest - validation croisée ===
MAE  : 0.838 jours
RMSE : 1.054 jours
R²   : 0.695

=== Random Forest - test set ===
MAE  : 0.860 jours
RMSE : 1.104 jours
R²   : 0.697


Paradoxalement, la Random Forest ne surpasse pas la régression linéaire. La complexité du modèle n'apporte donc pas automatiquement de gain de performance sur ce jeu de données.

## 12. XGBoost et optimisation Optuna

XGBoost est retenu comme candidat principal pour l'optimisation. Comme dans `readmission_model`, Optuna recherche les hyperparamètres sur le **train set uniquement**, avec une validation croisée à 5 folds.

La fonction objectif maximise indirectement la qualité de prédiction en minimisant la MAE.

In [47]:
def objective(trial):
    """Optimize XGBoost hyperparameters using cross-validated MAE."""
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 500
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 1.0, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10.0, log=True
        ),
    }

    trial_model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "regressor",
            XGBRegressor(
                **params,
                objective="reg:squarederror",
                eval_metric="mae",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ])

    scores = cross_val_score(
        trial_model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
    )

    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50)

best_params = study.best_params

print("Meilleurs hyperparamètres :")
display(pd.Series(best_params, name="value"))
print(f"MAE CV : {study.best_value:.3f} jours")

[I 2026-09-01 13:54:35,766] A new study created in memory with name: no-name-5ed87244-ad0e-484f-806e-2d50419d21c7
[I 2026-09-01 13:54:39,787] Trial 0 finished with value: 0.873304033279419 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 2.9152036385288193e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.873304033279419.
[I 2026-09-01 13:54:43,197] Trial 1 finished with value: 0.7997357487678528 and parameters: {'n_estimators': 341, 'max_depth': 8, 'learning_rate': 0.010636066512540286, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 2.8483918709107956e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 0.7997357487678528.
[I 2026-09-01 13:54:45,203] Trial 2 finished with value: 0.7931936740875244 and parameters: {'n_estimators': 222, 

Meilleurs hyperparamètres :


n_estimators        4.090000e+02
max_depth           2.000000e+00
learning_rate       5.393195e-02
subsample           6.565691e-01
colsample_bytree    9.977728e-01
min_child_weight    4.000000e+00
reg_alpha           4.773493e-05
reg_lambda          1.105554e-08
Name: value, dtype: float64

MAE CV : 0.759 jours


## 13. Modèle final XGBoost

Le modèle final est reconstruit avec les hyperparamètres sélectionnés par Optuna. Le jeu de test n'intervient qu'à cette étape finale.

In [48]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        XGBRegressor(
            **best_params,
            objective="reg:squarederror",
            eval_metric="mae",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

xgb_model.fit(X_train, y_train)
y_xgb_pred = xgb_model.predict(X_test)

xgb_test_mae = mean_absolute_error(y_test, y_xgb_pred)
xgb_test_rmse = root_mean_squared_error(
    y_test,
    y_xgb_pred,
)
xgb_test_r2 = r2_score(y_test, y_xgb_pred)

print("=== XGBoost optimisé - test set ===")
print(f"MAE  : {xgb_test_mae:.3f} jours")
print(f"RMSE : {xgb_test_rmse:.3f} jours")
print(f"R²   : {xgb_test_r2:.3f}")

=== XGBoost optimisé - test set ===
MAE  : 0.787 jours
RMSE : 1.051 jours
R²   : 0.725


Les performances obtenues en validation croisée sont proches de celles observées sur le jeu de test indépendant (écart de 0,028 jours). Cela suggère que le modèle généralise correctement et que l'optimisation n'a pas conduit à un surapprentissage manifeste.

In [49]:
feature_names = xgb_model.named_steps["preprocessor"].get_feature_names_out()
importances = xgb_model.named_steps["regressor"].feature_importances_

feature_importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance_df.head(15))

fig = px.bar(
    feature_importance_df.head(15),
    x="importance",
    y="feature",
    orientation="h",
    title="Feature importance native (XGBoost)",
    labels={"importance": "Importance", "feature": "Variable"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

,feature,importance
0,cat__primary_diagnosis_Hypertension,0.233743
1,cat__primary_diagnosis_Diabetes,0.154185
2,cat__primary_diagnosis_Stroke,0.136230
3,cat__primary_diagnosis_Sepsis,0.120989
4,cat__primary_diagnosis_Appendicitis,0.111465
5,cat__primary_diagnosis_Heart Failure,0.048536
6,cat__primary_diagnosis_COPD,0.033231
7,num__comorbidities_count,0.032361
8,cat__primary_diagnosis_Influenza,0.029146
9,cat__season_Winter,0.027188


La feature importance confirme l'importance de **primary_diagnosis**

## 14. Comparaison finale des modèles

La comparaison porte sur le même jeu de test et les mêmes métriques. La MAE est la métrique principale car elle reste directement interprétable en jours.

In [50]:
comparison = pd.DataFrame({
    "Model": [
        "DummyRegressor",
        "Baseline diagnostic",
        "Linear Regression",
        "Random Forest",
        "XGBoost optimisé",
    ],
    "MAE": [
        dummy_mae,
        diagnosis_mae,
        linear_test_mae,
        rf_test_mae,
        xgb_test_mae,
    ],
    "RMSE": [
        dummy_rmse,
        diagnosis_rmse,
        linear_test_rmse,
        rf_test_rmse,
        xgb_test_rmse,
    ],
    "R2": [
        dummy_r2,
        diagnosis_r2,
        linear_test_r2,
        rf_test_r2,
        xgb_test_r2,
    ],
})

display(comparison.round(3))

,Model,MAE,RMSE,R2
0,DummyRegressor,1.635,2.008,-0.003
1,Baseline diagnostic,0.947,1.204,0.640
2,Linear Regression,0.847,1.083,0.708
3,Random Forest,0.860,1.104,0.697
4,XGBoost optimisé,0.787,1.051,0.725


Le XGBoost optimisé par Optuna est le meilleur des modèles testés.

En effet, il bat la baseline diagnostic : 0,947 → 0,787 soit une amélioration absolue de 0,160 jour et environ 16,9 % d'amélioration relative de la MAE.

Il bat également la régression linéaire : 0,847 → 0,787 soit environ 7,1 % de réduction de l'erreur absolue moyenne.

Le modèle obtient un coefficient de détermination R² de 0,725, ce qui signifie qu'il explique environ 72,6 % de la variance de la durée de séjour observée dans le jeu de test.
La MAE de 0,787 signifie que la prédiction s'écarte d'environ 0,8 jours de la durée réelle, soit environ 10% de la durée moyenne. 

## 15. Analyse des erreurs par diagnostic

Une bonne performance globale peut masquer des difficultés sur certains profils. Nous comparons donc le modèle final à la baseline diagnostic pour chaque diagnostic.

In [51]:
final_results = X_test.copy()
final_results["actual"] = y_test
final_results["predicted"] = y_xgb_pred
final_results["baseline_pred"] = y_diagnosis_baseline

final_results["model_error"] = (
    final_results["actual"] - final_results["predicted"]
).abs()
final_results["baseline_error"] = (
    final_results["actual"] - final_results["baseline_pred"]
).abs()

final_by_diagnosis = (
    final_results.groupby("primary_diagnosis")
    [["model_error", "baseline_error"]]
    .mean()
    .rename(
        columns={
            "model_error": "mae_xgb",
            "baseline_error": "mae_baseline",
        }
    )
)

final_by_diagnosis["gain"] = (
    final_by_diagnosis["mae_baseline"]
    - final_by_diagnosis["mae_xgb"]
)

display(final_by_diagnosis.round(3))

,mae_xgb,mae_baseline,gain
primary_diagnosis,,,
Appendicitis,0.695,0.861,0.166
COPD,0.860,1.021,0.162
Diabetes,0.799,0.962,0.163
Fracture,0.667,0.847,0.180
Heart Failure,0.602,0.788,0.186
Hypertension,0.798,0.974,0.176
Influenza,0.823,0.858,0.035
Kidney Disease,0.830,0.953,0.123
Pneumonia,0.745,0.899,0.154


Le modèle apporte un meilleur résultat pour beaucoup de diagnostics. 
Par exemple :

- Heart Failure : gain de 0,196 j
- Stroke : 0,196 j
- Fracture : 0,189 j

En revanche pour l'influenza, le gain est seulement de 0,036 jour ce qui est très faible. Cela suggère que pour cette pathologie, connaitre le diagnostic fournit d'emblée l'essentiel de l'information prédictive disponible parmi les variables.  
Concernant le sepsis, qui présente la durée moyenne la plus élevée (10,76 jours), l'amélioration est de seulement 0,130 jour ce qui veut dire qu'une DMS élevée n'implique pas que la durée du séjour est plus facilement prédictible. 

In [52]:
final_results["residual"] = (
    final_results["actual"] - final_results["predicted"]
)

fig = px.scatter(
    final_results,
    x="actual",
    y="residual",
    color="primary_diagnosis",
    title="Analyse des résidus - XGBoost optimisé",
    labels={
        "actual": "Durée réelle (jours)",
        "residual": "Résidu (jours)",
        "primary_diagnosis": "Diagnostic",
    },
)
fig.add_hline(y=0, line_dash="dash")
fig.show()

### Lecture des résidus

Un résidu positif signifie que le modèle **sous-estime** la durée réelle ; un résidu négatif signifie qu'il la **surestime**.

Cette visualisation permet notamment de rechercher une structure dans les erreurs. Si les erreurs augmentent systématiquement avec la durée réelle, cela peut indiquer une difficulté à prédire correctement les séjours longs.

## 16. Suivi MLflow

Comme dans `readmission_model`, les paramètres, métriques et modèle final sont enregistrés dans MLflow.

Le point important est de conserver séparément les métriques de validation croisée et les métriques finales du jeu de test.

In [53]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="length_of_stay_model_xgb_mae"):
    mlflow.log_params({
        "model": "XGBRegressor",
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "cv_folds": CV_FOLDS,
        "optimization_metric": "MAE",
        **best_params,
    })

    mlflow.log_metrics({
        "dummy_test_mae": dummy_mae,
        "diagnosis_baseline_test_mae": diagnosis_mae,
        "linear_test_mae": linear_test_mae,
        "random_forest_test_mae": rf_test_mae,
        "xgb_test_mae": xgb_test_mae,
        "xgb_test_rmse": xgb_test_rmse,
        "xgb_test_r2": xgb_test_r2,
        "optuna_best_cv_mae": study.best_value,
    })

    mlflow.sklearn.log_model(
        xgb_model,
        name="xgboost_optimized",
        serialization_format="cloudpickle",
    )

print("✓ Run enregistré dans MLflow")

2026/09/01 13:56:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run length_of_stay_model_xgb_mae at: https://dagshub.com/quentin.seng67/hospital_readmission.mlflow/#/experiments/1/runs/38c86e91ffd946e48d6860b17c689763
🧪 View experiment at: https://dagshub.com/quentin.seng67/hospital_readmission.mlflow/#/experiments/1
✓ Run enregistré dans MLflow


## 17. Conclusion

Le modèle XGBoost optimisé est le meilleur modèle évalué pour prédire la durée d'hospitalisation, avec une MAE de 0,787 jour, un RMSE de 1,051 jour et un R² de 0,725 sur le jeu de test.  

La comparaison avec les baselines montre que primary_diagnosis constitue le principal facteur explicatif de la durée de séjour : une baseline fondée uniquement sur la durée moyenne par diagnostic atteint déjà une MAE de 0,947 jour. Les modèles de machine learning apportent cependant une amélioration supplémentaire en exploitant les caractéristiques individuelles des patients.  

La régression linéaire obtient une MAE de 0,847 jour, tandis que le Random Forest n'améliore pas ce résultat.  
L'optimisation d'XGBoost permet d'atteindre 0,787 jour, soit une réduction d'environ 16,9 % de l'erreur absolue moyenne par rapport à la baseline par diagnostic.  

La proximité entre la MAE obtenue en validation croisée (0,759 jour) et celle obtenue sur le jeu de test (0,787 jour) constitue un élément rassurant quant à la capacité de généralisation du modèle.  

Enfin, l'analyse par diagnostic montre que le gain du modèle varie selon la pathologie.  
Il est particulièrement important pour Heart Failure, Stroke et Fracture, alors qu'il est faible pour Influenza.  
Cela suggère que la capacité du modèle à individualiser la durée de séjour dépend du diagnostic considéré.